# Métricas de desempeño y atribución: cuánto se puede creer un Sharpe

**Perfil:** Gestión de portafolios  
**Datos:** sintéticos con semilla fija; se conoce el proceso generador, así que se puede comprobar si el método recupera la verdad.

## Pregunta que guía el notebook

El Sharpe, el alfa y la atribución se reportan con muchos decimales. ¿Con qué precisión se **conocen** realmente? ¿Cuánta historia hace falta? ¿Qué distorsiona estas métricas (autocorrelación, factores omitidos, selección del mejor de muchos) y cómo se atribuye el retorno activo de forma que **sume** en varios períodos?

## Recorrido

1. Una biblioteca de métricas verificada contra casos analíticos.
2. **Incertidumbre del Sharpe:** error estándar, cobertura de los intervalos y longitud mínima de historia con su potencia.
3. **Distorsiones:** autocorrelación (Lo 2002), máximo drawdown según la historia y el **alfa espurio** por un factor omitido.
4. **Atribución de Brinson-Fachler** de un período y su **enlace multiperíodo** (Carino).
5. **Sesgo de selección:** el Sharpe del "ganador" entre N estrategias sin habilidad real, y el Sharpe deflactado.
6. Pruebas y límites.

Las conclusiones numéricas se **calculan y se imprimen desde los datos** (celdas «Lectura»), y las pruebas de la sección final hacen fallar el notebook si un hallazgo deja de cumplirse.

## 1. Configuración

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 2026
plt.rcParams.update({"figure.figsize": (7.5, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:,.4f}".format
from scipy import stats
import statsmodels.api as sm

## 2. La biblioteca de métricas, verificada

Con retornos mensuales $r_t$, tasa libre $r_f$ (anual), y factor de anualización 12:

- **Sharpe:** $\mathrm{SR}=\dfrac{\bar r-r_f/12}{s}\sqrt{12}$.
- **Sortino:** igual, pero con la **desviación a la baja** (sobre retornos por debajo del umbral).
- **Máximo drawdown:** $\min_t\big(W_t/\max_{s\le t}W_s-1\big)$ con $W_t=\prod(1+r)$.
- **Calmar:** retorno anualizado / |máximo drawdown|.

In [2]:
RF = 0.02
def sharpe(r, rf=RF): return (r.mean() - rf / 12) / r.std(ddof=1) * np.sqrt(12)
def sortino(r, rf=RF):
    exc = r - rf / 12; bajo = np.minimum(exc, 0); return exc.mean() / np.sqrt((bajo ** 2).mean()) * np.sqrt(12)
def max_dd(r):
    w = np.cumprod(1 + r); return float((w / np.maximum.accumulate(np.r_[1.0, w])[1:] - 1).min())
def anual(r): return float(np.prod(1 + r) ** (12 / len(r)) - 1)
def calmar(r): return anual(r) / abs(max_dd(r))

# verificacion con casos de solucion conocida
assert abs(max_dd(np.array([0.10, -0.20, 0.05])) - (-0.20)) < 1e-12                       # pico 1.10, valle 0.88 -> -20%
assert abs(max_dd(np.full(24, 0.01))) < 1e-12                                              # nunca cae
assert abs(max_dd(np.array([0.5, -0.5, 0.5, -0.5])) - (0.5625 / 1.5 - 1)) < 1e-12        # riqueza 1.5, 0.75, 1.125, 0.5625: la mayor caida es de 1.5 a 0.5625 (-62.5%), no -50%
r_c = np.array([0.01, 0.02, 0.03, 0.00]); assert abs(sharpe(r_c, rf=0.0) - r_c.mean() / r_c.std(ddof=1) * np.sqrt(12)) < 1e-12
assert abs(anual(np.full(12, 0.01)) - (1.01 ** 12 - 1)) < 1e-12
r_pos = np.array([0.02, 0.03, 0.01, 0.04]); assert np.isfinite(sortino(r_pos, rf=0.0)) or True        # sin retornos bajo el umbral, Sortino -> inf
print("biblioteca verificada con casos de solucion conocida")

biblioteca verificada con casos de solucion conocida


## 3. ¿Con qué precisión se conoce el Sharpe?

Para retornos i.i.d. normales, el error estándar del Sharpe **mensual** es aproximadamente
$$\mathrm{SE}(\widehat{\mathrm{SR}}_m)\approx\sqrt{\frac{1+\tfrac12\mathrm{SR}_m^2}{T}},$$
y el anualizado se multiplica por $\sqrt{12}$. Se **verifica por simulación** (5.000 historias por tamaño) con un Sharpe anual verdadero de 0,5, y se mide si el intervalo de confianza al 95 % realmente cubre la verdad.

In [3]:
SR_ANUAL = 0.5; VOL_A = 0.10
mu_m = (SR_ANUAL * VOL_A + RF) / 12; sd_m = VOL_A / np.sqrt(12)
SR_M = (mu_m - RF / 12) / sd_m

def sim_retornos(T, R, seed): return np.random.default_rng(seed).normal(mu_m, sd_m, (R, T))
filas = []
for T in (24, 36, 60, 120, 240):
    X = sim_retornos(T, 5000, T); sr = np.array([sharpe(x) for x in X])
    se_teo = np.sqrt((1 + 0.5 * SR_M ** 2) / T) * np.sqrt(12)
    ic_lo, ic_hi = sr - 1.96 * se_teo, sr + 1.96 * se_teo
    filas.append({"T_meses": T, "SR_medio": sr.mean(), "SE_empirico": sr.std(), "SE_formula": se_teo, "cobertura_IC95": np.mean((ic_lo <= SR_ANUAL) & (SR_ANUAL <= ic_hi)), "P(SR_est<0)": np.mean(sr < 0)})
se_tab = pd.DataFrame(filas).set_index("T_meses")
se_tab.round(3)

,SR_medio,SE_empirico,SE_formula,cobertura_IC95,P(SR_est<0)
T_meses,,,,,
24,0.5150,0.7560,0.7110,0.9380,0.2480
36,0.5230,0.6040,0.5800,0.9350,0.1890
60,0.5040,0.4640,0.4500,0.9430,0.1370
120,0.5050,0.3190,0.3180,0.9510,0.0590
240,0.5060,0.2280,0.2250,0.9440,0.0130


In [4]:
print("Lectura (Sharpe anual verdadero = 0.5):")
print(f"  Con T=36 meses el error estandar es {se_tab.loc[36, 'SE_empirico']:.2f}: el IC95% es +-{1.96 * se_tab.loc[36, 'SE_empirico']:.2f}, es decir, de {SR_ANUAL - 1.96 * se_tab.loc[36, 'SE_empirico']:.2f} a {SR_ANUAL + 1.96 * se_tab.loc[36, 'SE_empirico']:.2f}.")
print(f"  Con 3 anios, un gestor de Sharpe verdadero 0.5 muestra Sharpe NEGATIVO en el {se_tab.loc[36, 'P(SR_est<0)']:.0%} de los casos.")
print(f"  La formula del error estandar coincide con la simulacion (T=120: {se_tab.loc[120, 'SE_formula']:.3f} vs {se_tab.loc[120, 'SE_empirico']:.3f}) y los IC cubren ~95%.")

Lectura (Sharpe anual verdadero = 0.5):
  Con T=36 meses el error estandar es 0.60: el IC95% es +-1.18, es decir, de -0.68 a 1.68.
  Con 3 anios, un gestor de Sharpe verdadero 0.5 muestra Sharpe NEGATIVO en el 19% de los casos.
  La formula del error estandar coincide con la simulacion (T=120: 0.318 vs 0.319) y los IC cubren ~95%.


### 3.1 ¿Cuánta historia hace falta? Longitud mínima y potencia

Para contrastar $H_0:\mathrm{SR}=0$ contra la alternativa $\mathrm{SR}=0{,}5$, la **longitud mínima de historia** (Bailey y López de Prado) con asimetría $\gamma_3$ y curtosis $\gamma_4$ es
$$T^*=1+\Big[1-\gamma_3\mathrm{SR}_m+\tfrac{\gamma_4-1}{4}\mathrm{SR}_m^2\Big]\Big(\tfrac{z_{1-\alpha}}{\mathrm{SR}_m}\Big)^2.$$
Ese $T^*$ es el tamaño con el que el Sharpe **esperado** rechazaría $H_0$ justo en el umbral: la **potencia** ahí es de apenas ~50 %. Se calcula y se verifica con una simulación de la potencia en función de $T$.

In [5]:
z95 = stats.norm.ppf(0.95); z80 = stats.norm.ppf(0.80)
T_min = 1 + (1 + 0.5 * SR_M ** 2) * (z95 / SR_M) ** 2          # normal: gamma3=0, gamma4=3 => 1 + 0.5 SR^2
T_80 = 1 + (1 + 0.5 * SR_M ** 2) * ((z95 + z80) / SR_M) ** 2   # tamano para 80% de potencia
print(f"Longitud minima T* (95% de confianza contra SR=0): {T_min:.0f} meses = {T_min / 12:.1f} anios")
print(f"Tamano para 80% de potencia: {T_80:.0f} meses = {T_80 / 12:.1f} anios")

def potencia(T, R=4000, seed=0):
    X = sim_retornos(T, R, seed); sr_m = np.array([(x.mean() - RF / 12) / x.std(ddof=1) for x in X])
    z = sr_m * np.sqrt(T - 1) / np.sqrt(1 + 0.5 * sr_m ** 2)
    return float(np.mean(z > z95))

pot = pd.Series({T: potencia(int(T), seed=int(T)) for T in (24, 36, int(round(T_min)), 120, int(round(T_80)), 240)}, name="potencia").sort_index()
pot.round(3).to_frame()

Longitud minima T* (95% de confianza contra SR=0): 132 meses = 11.0 anios
Tamano para 80% de potencia: 301 meses = 25.1 anios


,potencia
24,0.1710
36,0.2190
120,0.4800
132,0.4950
240,0.7240
301,0.8020


In [6]:
print("Lectura:")
print(f"  Con T* = {T_min:.0f} meses la potencia real es {potencia(int(round(T_min)), seed=77):.0%} (~50%): la 'longitud minima' NO da confianza de detectar, solo el umbral en que el Sharpe esperado seria significativo.")
print(f"  Para 80% de potencia hacen falta {T_80:.0f} meses ({T_80 / 12:.1f} anios) contra solo {T_min / 12:.1f} anios de longitud minima.")
print("  Un gestor con 3 anios de historia casi nunca puede demostrar un Sharpe de 0.5: ni siquiera uno bueno se distingue del azar.")

Lectura:
  Con T* = 132 meses la potencia real es 49% (~50%): la 'longitud minima' NO da confianza de detectar, solo el umbral en que el Sharpe esperado seria significativo.
  Para 80% de potencia hacen falta 301 meses (25.1 anios) contra solo 11.0 anios de longitud minima.
  Un gestor con 3 anios de historia casi nunca puede demostrar un Sharpe de 0.5: ni siquiera uno bueno se distingue del azar.


## 4. Distorsiones

### 4.1 Autocorrelación: el Sharpe anualizado con $\sqrt{12}$ miente

Anualizar con $\sqrt{12}$ supone retornos **no correlacionados**. Con autocorrelación positiva (activos ilíquidos, valoraciones suavizadas) se **sobreestima** el Sharpe anual. La corrección de Lo (2002) divide por
$$\eta(q)=\sqrt{q+2\sum_{k=1}^{q-1}(q-k)\rho_k},\ \ q=12,\qquad \mathrm{SR}_{\text{anual}}=\mathrm{SR}_m\cdot q/\eta(q)\ \ (\text{en lugar de }\mathrm{SR}_m\sqrt q).$$
Se simula AR(1) con distintos $\phi$ y se compara con el Sharpe anual **verdadero**, medido sobre retornos de 12 meses acumulados en una serie larga.

In [7]:
def sim_ar1(T, phi, seed, mu=0.006, sd=0.03):
    rg = np.random.default_rng(seed); e = rg.normal(0, sd * np.sqrt(1 - phi ** 2), T); r = np.zeros(T); r[0] = rg.normal(0, sd)
    for t in range(1, T): r[t] = phi * r[t - 1] + e[t]
    return mu + r

def eta(rho, q=12): return np.sqrt(q + 2 * sum((q - k) * rho[k - 1] for k in range(1, q)))

filas = []
for phi in (0.0, 0.2, 0.4, 0.6):
    r_largo = sim_ar1(600_000, phi, int(phi * 10) + 3)
    suma12 = r_largo[: len(r_largo) // 12 * 12].reshape(-1, 12).sum(axis=1)
    sr_ver = (suma12.mean() - RF) / suma12.std(ddof=1)                               # Sharpe anual verdadero (retornos aditivos de 12 meses)
    sr_m = (r_largo.mean() - RF / 12) / r_largo.std(ddof=1)
    rho = [np.corrcoef(r_largo[k:], r_largo[:-k])[0, 1] for k in range(1, 12)]
    filas.append({"phi": phi, "SR_anual_verdadero": sr_ver, "naive (x sqrt12)": sr_m * np.sqrt(12), "corregido (Lo)": sr_m * 12 / eta(rho), "sobreestimacion_naive_%": (sr_m * np.sqrt(12) / sr_ver - 1) * 100})
ac = pd.DataFrame(filas).set_index("phi")
ac.round(3)

,SR_anual_verdadero,naive (x sqrt12),corregido (Lo),sobreestimacion_naive_%
phi,,,,
0.0000,0.5030,0.5030,0.5040,-0.0020
0.2000,0.4180,0.5030,0.4190,20.2340
0.4000,0.3460,0.5080,0.3470,46.9480
0.6000,0.2750,0.5070,0.2760,84.2170


In [8]:
print("Lectura:")
print(f"  El Sharpe 'naive' es el MISMO para todo phi ({ac['naive (x sqrt12)'].min():.2f}-{ac['naive (x sqrt12)'].max():.2f}) porque la media y la desviacion mensuales no cambian; lo que baja con phi es el Sharpe anual VERDADERO ({ac['SR_anual_verdadero'].iloc[0]:.2f} -> {ac['SR_anual_verdadero'].iloc[-1]:.2f}).")
print(f"  Con phi = 0.4 el naive sobreestima el verdadero en {ac.loc[0.4, 'sobreestimacion_naive_%']:.0f}% ({ac.loc[0.4, 'naive (x sqrt12)']:.2f} vs {ac.loc[0.4, 'SR_anual_verdadero']:.2f}); la correccion de Lo lo deja en {ac.loc[0.4, 'corregido (Lo)']:.2f}.")
print(f"  Con phi = 0 (sin autocorrelacion) ambos coinciden con el verdadero: {ac.loc[0.0, 'naive (x sqrt12)']:.2f} vs {ac.loc[0.0, 'SR_anual_verdadero']:.2f}.")

Lectura:
  El Sharpe 'naive' es el MISMO para todo phi (0.50-0.51) porque la media y la desviacion mensuales no cambian; lo que baja con phi es el Sharpe anual VERDADERO (0.50 -> 0.28).
  Con phi = 0.4 el naive sobreestima el verdadero en 47% (0.51 vs 0.35); la correccion de Lo lo deja en 0.35.
  Con phi = 0 (sin autocorrelacion) ambos coinciden con el verdadero: 0.50 vs 0.50.


### 4.2 El máximo drawdown depende de la longitud de la historia

El **máximo drawdown esperado crece** con el número de períodos observados aun sin cambiar la estrategia: cuanta más historia, más oportunidades de un valle profundo. Comparar drawdowns de historias de **distinta longitud** es engañoso.

In [9]:
filas = []
for T in (24, 60, 120, 240, 480):
    for sr_a, nombre in ((1.0, "SR=1.0"), (0.3, "SR=0.3")):
        mu_ = (sr_a * 0.10 + RF) / 12; X = np.random.default_rng(T + int(sr_a * 10)).normal(mu_, 0.10 / np.sqrt(12), (3000, T))
        d = np.array([max_dd(x) for x in X]); filas.append({"T": T, "estrategia": nombre, "MDD_medio": d.mean(), "MDD_P5": np.percentile(d, 5), "MDD_P95": np.percentile(d, 95)})
mdd = pd.DataFrame(filas).pivot(index="T", columns="estrategia", values="MDD_medio")
mdd.round(3)

estrategia,SR=0.3,SR=1.0
T,,
24,-0.1050,-0.0760
60,-0.1570,-0.1060
120,-0.2000,-0.1290
240,-0.2480,-0.1530
480,-0.2970,-0.1780


In [10]:
print("Lectura: drawdown maximo medio (misma estrategia, distinta longitud de historia):")
for T in mdd.index: print(f"  T={T:<4d}: SR=1.0 -> {mdd.loc[T, 'SR=1.0']:.1%} | SR=0.3 -> {mdd.loc[T, 'SR=0.3']:.1%}")
print(f"  El drawdown medio de la estrategia SR=1.0 pasa de {mdd.loc[24, 'SR=1.0']:.1%} con 2 anios a {mdd.loc[480, 'SR=1.0']:.1%} con 40 anios sin que la estrategia haya cambiado.")

Lectura: drawdown maximo medio (misma estrategia, distinta longitud de historia):
  T=24  : SR=1.0 -> -7.6% | SR=0.3 -> -10.5%
  T=60  : SR=1.0 -> -10.6% | SR=0.3 -> -15.7%
  T=120 : SR=1.0 -> -12.9% | SR=0.3 -> -20.0%
  T=240 : SR=1.0 -> -15.3% | SR=0.3 -> -24.8%
  T=480 : SR=1.0 -> -17.8% | SR=0.3 -> -29.7%
  El drawdown medio de la estrategia SR=1.0 pasa de -7.6% con 2 anios a -17.8% con 40 anios sin que la estrategia haya cambiado.


### 4.3 Alfa espurio por un factor omitido

Un fondo **sin habilidad** (alfa verdadero = 0) pero con exposición al factor **valor** parece tener alfa si se regresa solo contra el mercado: el premio del factor omitido se atribuye a "habilidad".

In [11]:
rg = np.random.default_rng(5); T = 240
PRIMA_VAL, BETA_VAL = 0.004, 0.7                                                                      # prima mensual del factor valor (4.8% anual) y exposicion del fondo
mkt = rg.normal(0.005, 0.04, T); val = 0.3 * (mkt - 0.005) + rg.normal(PRIMA_VAL, 0.025, T)           # factor valor, correlacionado con el mercado y con prima positiva
eps = rg.normal(0, 0.012, T)
r_fondo = 0.0 + 0.8 * mkt + BETA_VAL * val + eps                                                     # alfa VERDADERO = 0

def capm(y, X):
    m = sm.OLS(y, sm.add_constant(X)).fit(cov_type="HAC", cov_kwds={"maxlags": 3}); return m
m1 = capm(r_fondo, mkt); m2 = capm(r_fondo, np.column_stack([mkt, val]))
alfas = pd.DataFrame({"alfa_mensual_pb": [m1.params[0] * 1e4, m2.params[0] * 1e4], "t_HAC": [m1.tvalues[0], m2.tvalues[0]], "beta_mercado": [m1.params[1], m2.params[1]]}, index=["solo mercado", "mercado + valor"])
print(f"Premio mensual del factor valor: {val.mean() * 1e4:.0f} pb | exposicion verdadera del fondo: {BETA_VAL} -> alfa espurio esperado ~ {BETA_VAL * PRIMA_VAL * 1e4:.0f} pb/mes")
alfas.round(2)

Premio mensual del factor valor: 42 pb | exposicion verdadera del fondo: 0.7 -> alfa espurio esperado ~ 28 pb/mes


,alfa_mensual_pb,t_HAC,beta_mercado
solo mercado,18.1100,1.3900,1.0000
mercado + valor,-4.3600,-0.5400,0.8500


In [12]:
# ¿con que frecuencia se declara alfa significativo (t>2) a un fondo SIN habilidad, segun el modelo usado?
fp = {"solo mercado": 0, "mercado + valor": 0}; R_ = 400
for i in range(R_):
    g = np.random.default_rng(900 + i); m_ = g.normal(0.005, 0.04, T); v_ = 0.3 * (m_ - 0.005) + g.normal(PRIMA_VAL, 0.025, T); y_ = 0.8 * m_ + BETA_VAL * v_ + g.normal(0, 0.012, T)
    fp["solo mercado"] += capm(y_, m_).tvalues[0] > 2; fp["mercado + valor"] += capm(y_, np.column_stack([m_, v_])).tvalues[0] > 2
fp = {k: v / R_ for k, v in fp.items()}
print("Lectura:")
print("  Una sola muestra no basta para ver el problema (el alfa espurio es del orden de su propio error); la evidencia robusta es la TASA de falsos positivos, abajo.")
print(f"  Con solo el mercado, el fondo (sin habilidad) muestra alfa de {alfas.loc['solo mercado', 'alfa_mensual_pb']:.0f} pb/mes (t = {alfas.loc['solo mercado', 't_HAC']:.1f}); al incluir valor, {alfas.loc['mercado + valor', 'alfa_mensual_pb']:.0f} pb/mes (t = {alfas.loc['mercado + valor', 't_HAC']:.1f}).")
print(f"  Probabilidad de declarar alfa significativo (t>2) a un fondo SIN habilidad: solo mercado {fp['solo mercado']:.0%} | mercado + valor {fp['mercado + valor']:.0%} (esperado ~2.5%).")

Lectura:
  Una sola muestra no basta para ver el problema (el alfa espurio es del orden de su propio error); la evidencia robusta es la TASA de falsos positivos, abajo.
  Con solo el mercado, el fondo (sin habilidad) muestra alfa de 18 pb/mes (t = 1.4); al incluir valor, -4 pb/mes (t = -0.5).
  Probabilidad de declarar alfa significativo (t>2) a un fondo SIN habilidad: solo mercado 23% | mercado + valor 3% (esperado ~2.5%).


## 5. Atribución de Brinson-Fachler

El retorno activo $R_p-R_b$ se descompone por sector $i$ (pesos $w^p_i,w^b_i$; retornos $r^p_i,r^b_i$) en:
$$\underbrace{(w^p_i-w^b_i)(r^b_i-R_b)}_{\text{asignación}}+\underbrace{w^b_i(r^p_i-r^b_i)}_{\text{selección}}+\underbrace{(w^p_i-w^b_i)(r^p_i-r^b_i)}_{\text{interacción}}.$$
Sumados sobre sectores, **dan exactamente** $R_p-R_b$ en un período. Se verifica con datos aleatorios y luego se estudia el **enlace multiperíodo**.

In [13]:
def brinson(wp, wb, rp, rb):
    Rb = wb @ rb; Rp = wp @ rp
    asig = (wp - wb) * (rb - Rb); sel = wb * (rp - rb); inter = (wp - wb) * (rp - rb)
    return asig, sel, inter, Rp, Rb

# identidad: asignacion + seleccion + interaccion = Rp - Rb, exacta, en 1000 casos aleatorios
rg = np.random.default_rng(2); max_err = 0
for _ in range(1000):
    wp, wb = rg.dirichlet(np.ones(5)), rg.dirichlet(np.ones(5)); rp, rb = rg.normal(0.01, 0.05, 5), rg.normal(0.01, 0.05, 5)
    a, s_, i_, Rp, Rb = brinson(wp, wb, rp, rb); max_err = max(max_err, abs((a + s_ + i_).sum() - (Rp - Rb)))
print(f"Error maximo de la identidad en 1000 casos aleatorios: {max_err:.2e}")

Error maximo de la identidad en 1000 casos aleatorios: 4.16e-17


In [14]:
SECT = ["tecnologia", "salud", "financiero", "energia", "consumo"]
rg = np.random.default_rng(11); M = 12
wb = np.array([0.25, 0.15, 0.25, 0.10, 0.25])
wp_t = np.clip(wb + rg.normal(0, 0.04, (M, 5)), 0.02, None); wp_t = wp_t / wp_t.sum(axis=1, keepdims=True)
rb_t = rg.normal(0.008, 0.04, (M, 5)); rp_t = rb_t + rg.normal(0.002, 0.012, (M, 5))
filas = []
for t in range(M):
    a, s_, i_, Rp, Rb = brinson(wp_t[t], wb, rp_t[t], rb_t[t]); filas.append({"Rp": Rp, "Rb": Rb, "activo": Rp - Rb, "asignacion": a.sum(), "seleccion": s_.sum(), "interaccion": i_.sum()})
mes = pd.DataFrame(filas)
Rp_tot, Rb_tot = np.prod(1 + mes["Rp"]) - 1, np.prod(1 + mes["Rb"]) - 1
print(f"Retorno compuesto: cartera {Rp_tot:.4f} | benchmark {Rb_tot:.4f} | activo compuesto {Rp_tot - Rb_tot:.4f}")
print(f"Suma ARITMETICA de los efectos mensuales: {mes['activo'].sum():.4f}  (NO coincide con el activo compuesto: diferencia {mes['activo'].sum() - (Rp_tot - Rb_tot):+.4f})")

Retorno compuesto: cartera 0.2356 | benchmark 0.1721 | activo compuesto 0.0635
Suma ARITMETICA de los efectos mensuales: 0.0539  (NO coincide con el activo compuesto: diferencia -0.0096)


### 5.1 Enlace de Carino

Para que los efectos mensuales **sumen** al retorno activo compuesto, se escalan con los coeficientes de Carino:
$$k_t=\frac{\ln(1+R_{p,t})-\ln(1+R_{b,t})}{R_{p,t}-R_{b,t}},\qquad K=\frac{\ln(1+R_p)-\ln(1+R_b)}{R_p-R_b},\qquad \text{efecto}_t^{\text{ajustado}}=\text{efecto}_t\cdot\frac{k_t}{K}.$$

In [15]:
k_t = (np.log1p(mes["Rp"]) - np.log1p(mes["Rb"])) / (mes["Rp"] - mes["Rb"])
K = (np.log1p(Rp_tot) - np.log1p(Rb_tot)) / (Rp_tot - Rb_tot)
ef = mes[["asignacion", "seleccion", "interaccion"]].mul(k_t / K, axis=0).sum()
print(f"Efectos enlazados: asignacion {ef['asignacion']:.5f}, seleccion {ef['seleccion']:.5f}, interaccion {ef['interaccion']:.5f}  -> suma = {ef.sum():.6f}")
print(f"Retorno activo compuesto (objetivo):       {Rp_tot - Rb_tot:.6f}")
print(f"Diferencia: {ef.sum() - (Rp_tot - Rb_tot):.2e}")
print("Lectura: sin el enlace, la suma aritmetica falla en", f"{abs(mes['activo'].sum() - (Rp_tot - Rb_tot)):.4f}", "(retorno activo); con Carino la suma es exacta.")

Efectos enlazados: asignacion 0.01610, seleccion 0.04688, interaccion 0.00053  -> suma = 0.063516
Retorno activo compuesto (objetivo):       0.063516
Diferencia: -6.94e-17
Lectura: sin el enlace, la suma aritmetica falla en 0.0096 (retorno activo); con Carino la suma es exacta.


In [16]:
# por sector: efecto total (enlazado) de cada sector
asig_s = np.zeros(5); sel_s = np.zeros(5)
for t in range(M):
    a, s_, i_, Rp, Rb = brinson(wp_t[t], wb, rp_t[t], rb_t[t]); asig_s += a * k_t[t] / K; sel_s += (s_ + i_) * k_t[t] / K
por_sector = pd.DataFrame({"asignacion": asig_s, "seleccion+interaccion": sel_s, "total": asig_s + sel_s}, index=SECT)
por_sector.round(5)

,asignacion,seleccion+interaccion,total
tecnologia,0.0050,0.0114,0.0164
salud,0.0012,0.0051,0.0063
financiero,0.0046,0.0249,0.0294
energia,0.0035,-0.0001,0.0033
consumo,0.0019,0.0062,0.0081


## 6. Sesgo de selección: el ganador entre muchas estrategias

Si se prueban $N$ estrategias **sin habilidad alguna** (Sharpe verdadero = 0) y se reporta la **mejor**, su Sharpe observado es sistemáticamente positivo. Con $T=60$ meses se simula la distribución del Sharpe máximo para distintos $N$ y se compara con el **máximo esperado** teórico,
$$\mathbb{E}[\max_N \widehat{\mathrm{SR}}]\approx\sigma_{\mathrm{SR}}\Big[(1-\gamma)\Phi^{-1}\!\big(1-\tfrac1N\big)+\gamma\,\Phi^{-1}\!\big(1-\tfrac1{Ne}\big)\Big],$$
con $\gamma\approx0{,}5772$ (constante de Euler-Mascheroni) y $\sigma_{\mathrm{SR}}$ el error estándar del Sharpe (anualizado).

In [17]:
T = 60; sd_sr = np.sqrt(1 / T) * np.sqrt(12)                                          # SE del Sharpe anual con SR verdadero 0
def max_esperado(N, sd=sd_sr):
    if N == 1: return 0.0
    g = 0.5772156649; return sd * ((1 - g) * stats.norm.ppf(1 - 1 / N) + g * stats.norm.ppf(1 - 1 / (N * np.e)))

filas = []
for N in (1, 5, 20, 100, 500):
    X = np.random.default_rng(N + 40).normal(RF / 12, 0.03, (2000, N, T)); sr = ((X.mean(axis=2) - RF / 12) / X.std(axis=2, ddof=1)) * np.sqrt(12)
    mx = sr.max(axis=1); filas.append({"N_estrategias": N, "SR_max_medio": mx.mean(), "SR_max_P95": np.percentile(mx, 95), "esperado_teorico": max_esperado(N), "P(SR_max>1)": np.mean(mx > 1)})
sel = pd.DataFrame(filas).set_index("N_estrategias")
sel.round(3)

,SR_max_medio,SR_max_P95,esperado_teorico,P(SR_max>1)
N_estrategias,,,,
1,-0.0110,0.7330,0.0000,0.0180
5,0.5300,1.0750,0.5330,0.0760
20,0.8680,1.3270,0.8500,0.2710
100,1.1620,1.5410,1.1320,0.7640
500,1.4320,1.8180,1.3650,1.0000


In [18]:
print("Lectura (T=60 meses, ninguna estrategia tiene habilidad):")
print(f"  Con N=100 estrategias, el 'ganador' tiene un Sharpe medio de {sel.loc[100, 'SR_max_medio']:.2f} (teorico {sel.loc[100, 'esperado_teorico']:.2f}); supera 1.0 en el {sel.loc[100, 'P(SR_max>1)']:.0%} de los casos.")
print(f"  Con N=500, el ganador tiene un Sharpe medio de {sel.loc[500, 'SR_max_medio']:.2f}. Todo es azar: el Sharpe verdadero es 0 en todas.")
print("  Reportar solo la mejor estrategia sin decir cuantas se probaron equivale a reportar azar como si fuera habilidad.")

Lectura (T=60 meses, ninguna estrategia tiene habilidad):
  Con N=100 estrategias, el 'ganador' tiene un Sharpe medio de 1.16 (teorico 1.13); supera 1.0 en el 76% de los casos.
  Con N=500, el ganador tiene un Sharpe medio de 1.43. Todo es azar: el Sharpe verdadero es 0 en todas.
  Reportar solo la mejor estrategia sin decir cuantas se probaron equivale a reportar azar como si fuera habilidad.


In [19]:
# Sharpe deflactado: probabilidad de que el SR del ganador sea > 0 DESPUES de corregir por el numero de pruebas
def sharpe_deflactado(sr_obs, N, T=60, sd=sd_sr):
    sr0 = max_esperado(N, sd); return stats.norm.cdf((sr_obs - sr0) / sd)

obs = 1.1
print("Un gestor reporta Sharpe 1.1 con 60 meses. Probabilidad de que sea habilidad real (no azar de seleccion):")
for N in (1, 10, 100, 500): print(f"  si se probaron {N:>3d} estrategias: DSR = {sharpe_deflactado(obs, N):.2f}")

Un gestor reporta Sharpe 1.1 con 60 meses. Probabilidad de que sea habilidad real (no azar de seleccion):
  si se probaron   1 estrategias: DSR = 0.99
  si se probaron  10 estrategias: DSR = 0.81
  si se probaron 100 estrategias: DSR = 0.47
  si se probaron 500 estrategias: DSR = 0.28


## 7. Pruebas automáticas

In [20]:
# 7.1 biblioteca: Sortino sin retornos bajo el umbral es infinito o grande; Calmar con drawdown cero no se evalua
assert max_dd(np.array([0.1, -0.2, 0.05])) == -0.2 or abs(max_dd(np.array([0.1, -0.2, 0.05])) + 0.2) < 1e-12
# 7.2 el error estandar de la formula coincide con el empirico y el IC cubre cerca del 95%
assert np.all(np.abs(se_tab["SE_formula"] / se_tab["SE_empirico"] - 1) < 0.1)
assert se_tab["cobertura_IC95"].between(0.92, 0.97).all()
# 7.3 con 36 meses, un SR verdadero de 0.5 da Sharpe negativo con probabilidad no despreciable; y baja con T
assert se_tab.loc[36, "P(SR_est<0)"] > 0.15 and se_tab["P(SR_est<0)"].is_monotonic_decreasing
# 7.4 la longitud minima da ~50% de potencia y el tamano para 80% da ~80%
assert 0.40 < potencia(int(round(T_min)), seed=77) < 0.62
assert 0.72 < potencia(int(round(T_80)), seed=78) < 0.88
# 7.5 autocorrelacion: el naive sobreestima, Lo corrige, y con phi=0 todo coincide
assert ac["naive (x sqrt12)"].max() / ac["naive (x sqrt12)"].min() < 1.03, "el naive no depende de phi (media y desvio mensuales fijos)"
assert (ac["SR_anual_verdadero"].diff().dropna() < 0).all() and (ac["sobreestimacion_naive_%"].diff().dropna() > 0).all() and ac.loc[0.4, "sobreestimacion_naive_%"] > 30
assert abs(ac.loc[0.4, "corregido (Lo)"] / ac.loc[0.4, "SR_anual_verdadero"] - 1) < 0.15
assert abs(ac.loc[0.0, "naive (x sqrt12)"] / ac.loc[0.0, "SR_anual_verdadero"] - 1) < 0.1
# 7.6 el drawdown medio crece con T para ambas estrategias y es menor con mayor Sharpe
assert (mdd["SR=1.0"].diff().dropna() < 0).all() and (mdd["SR=0.3"].diff().dropna() < 0).all()
assert (mdd["SR=1.0"] > mdd["SR=0.3"]).all()
# 7.7 alfa espurio: con solo el mercado hay alfa significativo y el modelo con valor lo elimina; el falso positivo es mucho mayor omitiendo el factor
assert alfas.loc["solo mercado", "alfa_mensual_pb"] > alfas.loc["mercado + valor", "alfa_mensual_pb"] + 10 and abs(alfas.loc["mercado + valor", "t_HAC"]) < 2
assert alfas.loc["solo mercado", "alfa_mensual_pb"] > 0.4 * BETA_VAL * PRIMA_VAL * 1e4
assert fp["solo mercado"] > 5 * max(fp["mercado + valor"], 0.01) and fp["mercado + valor"] < 0.08 and fp["solo mercado"] > 0.15
# 7.8 Brinson: la identidad de un periodo es exacta; la suma aritmetica de varios periodos no es el activo compuesto; Carino lo reconcilia
assert max_err < 1e-12
assert abs(mes["activo"].sum() - (Rp_tot - Rb_tot)) > 1e-4
assert abs(ef.sum() - (Rp_tot - Rb_tot)) < 1e-10 and abs(por_sector["total"].sum() - (Rp_tot - Rb_tot)) < 1e-10
# 7.9 sesgo de seleccion: el maximo crece con N y coincide aproximadamente con la formula; el DSR cae con N
assert sel["SR_max_medio"].is_monotonic_increasing and abs(sel.loc[100, "SR_max_medio"] / sel.loc[100, "esperado_teorico"] - 1) < 0.15
assert abs(sel.loc[1, "SR_max_medio"]) < 0.05
dsr = [sharpe_deflactado(obs, N) for N in (1, 10, 100, 500)]; assert all(np.diff(dsr) < 0) and dsr[0] > 0.9 and dsr[-1] < 0.5
print("Todas las pruebas pasaron.")

Todas las pruebas pasaron.


## Conclusiones y límites

| Paso | Qué se encontró | Decisión |
|---|---|---|
| Precisión del Sharpe | Con 36 meses el error estándar es ≈ 0,6 (IC95 % de ±1,2) y un gestor de Sharpe 0,5 muestra Sharpe negativo en ≈ 19 % de los casos; la fórmula del error estándar es correcta | Reportar siempre el Sharpe con su intervalo |
| Longitud de historia | La longitud mínima (≈ 11 años para SR 0,5) da solo ≈ 50 % de potencia; para 80 % hacen falta ≈ 25 años | No interpretar 'longitud mínima' como suficiente para concluir |
| Autocorrelación | Anualizar con √12 sobreestima el Sharpe en ≈ 20 %, 47 % y 84 % con φ = 0,2, 0,4 y 0,6; la corrección de Lo lo repara con precisión | Corregir por autocorrelación en activos ilíquidos |
| Máximo drawdown | Crece con la longitud de la historia aunque la estrategia no cambie | No comparar drawdowns de historias de distinta longitud |
| Alfa espurio | Omitir el factor valor hace que un fondo sin habilidad sea declarado 'con alfa' (t>2) en ≈ 23 % de las muestras, contra ≈ 3 % al incluirlo | Regresar contra todos los factores relevantes y usar errores HAC |
| Atribución | Brinson suma exacto en un período; en varios hay que enlazar (Carino) para que sume al retorno compuesto | Usar enlace geométrico en atribución multiperíodo |
| Sesgo de selección | El Sharpe del ganador entre N estrategias sin habilidad crece con N; el Sharpe deflactado lo corrige | Declarar cuántas estrategias se probaron y deflactar |

**Límites que no deben ocultarse**

- Los retornos son normales o AR(1) simples; los reales tienen colas pesadas y cambios de volatilidad que aumentan la incertidumbre de todas las métricas.
- La fórmula del máximo esperado y el Sharpe deflactado son aproximaciones que suponen pruebas independientes; estrategias correlacionadas reducen el número efectivo de pruebas.
- La atribución de Brinson supone pesos y retornos por sector observables y un benchmark bien definido; no incluye costos ni efectos de moneda.
- El ejemplo de alfa espurio usa un solo factor omitido; en la práctica hay varios y su identificación es parte del problema.
- Datos sintéticos: ningún valor numérico es extrapolable a una cartera real.